# Day 2 Homework: Day 1 Summarizer on Ollama

The Day 1 webpage summarizer, but running against an open-source model on this machine via Ollama's OpenAI-compatible endpoint.

Only two things change from Day 1: the client's `base_url`/`api_key`, and the model name. No API charges, and the page content never leaves your machine.

In [2]:
# imports

import requests
from scraper import fetch_website_contents
from IPython.display import Markdown, display, update_display
from openai import OpenAI

## Connect to Ollama

The Ollama Mac app runs the server in the background, so there's no need for `ollama serve`. This cell should print `b'Ollama is running'`.

In [3]:
OLLAMA_BASE_URL = "http://localhost:11434/v1"
MODEL = "llama3.2"

requests.get("http://localhost:11434").content

b'Ollama is running'

In [4]:
# Ollama ignores the key, but the client requires a non-empty one

ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama")

# Make sure the model is pulled
pulled = [m.id for m in ollama.models.list().data]
print(pulled)
if not any(name.split(":")[0] == MODEL.split(":")[0] for name in pulled):
    print(f"{MODEL} not found - run: ollama pull {MODEL}")

['deepseek-r1:1.5b', 'llama3.2:latest', 'phi3:latest', 'gemma3:270m']


## Prompts and messages (same as Day 1)

In [5]:
system_prompt = """
You are a snarky assistant that analyzes the contents of a website,
and provides a short, snarky, humorous summary, ignoring text that might be navigation related.
Respond in markdown. Do not wrap the markdown in a code block - respond just with the markdown.
"""

user_prompt_prefix = """
Here are the contents of a website.
Provide a short summary of this website.
If it includes news or announcements, then summarize these too.

"""

In [6]:
def messages_for(website):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_prefix + website}
    ]

## Summarize - the only real change is `ollama` instead of `openai`

In [8]:
def summarize(url, model=MODEL):
    website = fetch_website_contents(url)
    response = ollama.chat.completions.create(
        model=model,
        messages=messages_for(website)
    )
    return response.choices[0].message.content

In [9]:
def display_summary(url, model=MODEL):
    summary = summarize(url, model)
    display(Markdown(summary))

In [10]:
display_summary("https://edwarddonner.com")

# A self-absorbed AI enthusiast's website

Meet Edward Donner, CEO of Nebula.io, former founder of AI startup untapt acquired in 2021, and a dude who's really, really into Large Language Models (LLMs). His website is basically a love letter to himself, with a "Learn AI with me" tone and a plethora of Udemy courses to boot. Oh, and he's also into amateur music production and Hacker News. News-wise, his AI courses are gaining popularity, with 900,000 enrollments and best-seller tags.

In [11]:
display_summary("https://anthropic.com")

`Anthropic aims to harness the benefits of AI while mitigating its risks as a public benefit corporation. Their products, including Sonnet 5.5, focus on safety and security at the "frontier" of AI advancements.`

`: Announcement: Sonnet 5.5 is out, boasting 30% faster performance and cost savings. The latest release hits on September 28, 2026.`

## Bonus: streaming

Local models are slower than the cloud APIs, so streaming the tokens as they arrive feels much snappier.

In [12]:
def stream_summary(url, model=MODEL):
    website = fetch_website_contents(url)
    stream = ollama.chat.completions.create(
        model=model,
        messages=messages_for(website),
        stream=True
    )
    response = ""
    handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ""
        update_display(Markdown(response), display_id=handle.display_id)

In [13]:
stream_summary("https://edwarddonner.com")

*Welcome to Edward Donner's website, where he loudly speaks about AI and gets praised for being "hilariously knowledgeable"* 

Apparently, he founded an AI startup (Nebula.io), made another AI startup disappear (untapt, you're missed), and decided to share his expertise through Udemy courses on AI (specifically LLMs), which somehow became best-sellers. Don't worry, he'll happily chat with you and your digital avatar.

**News/Miscellaneous:**
* Edward is now the king of AI blogs (his website).
* AI course "Which order to take the AI courses?" is, well, unclear. 
* A few AI-related resources for "Vibe Coder" and "Agentic Engineers"

## Bonus: compare model sizes

The homework's "disadvantage" in action: try the same page with a tiny model. Any model you've pulled works here (`ollama list` shows them).

In [14]:
for model in ["gemma3:270m", "llama3.2"]:
    display(Markdown(f"### {model}"))
    display_summary("https://edwarddonner.com", model)

### gemma3:270m

Markdown

### llama3.2

### Short Summary:

This website is a personal blog or landing page, seemingly run by Edward Donner, a self-proclaimed "LLM expert" and AI enthusiast. The site offers tutorials, courses, and blog posts on the topic, as well as updates on Donner's personal projects and ventures, like his AI startup Nebula.io and Udemy courses.

### News/Announcements:

* Edward Donner recently released new resources (February 17, 2026, January 4, 2026, September 15, 2025, and May 28, 2025), including a guide on taking the AI courses in the correct order.